# Fine tune to follow instructions

## Stage1: preparing dataset

In [1]:
import json
import os
import urllib

file_path = 'instruction_data.json'
url = (
    'https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/refs/heads/main/ch07/01_main-chapter-code/instruction-data.json'
)

if not os.path.exists(file_path):
    with urllib.request.urlopen(url) as response:
        text_data = response.read().decode('utf-8')
else:
    with open(file_path, 'r', encoding='utf-8') as f:
        text_data = f.read()

data = json.loads(text_data)
print(len(data))

1100


In [2]:
# using Alpaca-style format
def format_input(entry):
    instruction_text = ('Below is an instruction that describe a task.'
                        'Write a response that appropriately completes the request.'
                        f"\n\\n ### Instruction: \\n{entry['instruction']}")
    input_text = (f"\n\\n### Input: \\n{entry['input']}" if entry['input'] else "")

    return instruction_text + input_text

input = format_input(data[0])
response = f"\n\\n### Response: \\n{data[0]['output']}"
print(input + response)

Below is an instruction that describe a task.Write a response that appropriately completes the request.
\n ### Instruction: \nEvaluate the following phrase by transforming it into the spelling given.
\n### Input: \nfreind --> friend
\n### Response: \nThe spelling of the given phrase "freind" is incorrect, the correct spelling is "friend".


In [3]:
# train,val,test split
train_portion = int(len(data)*0.85)
test_portion = int(len(data)*0.1)

train_data = data[:train_portion]
test_data = data[train_portion: train_portion + test_portion]
val_data = data[train_portion + test_portion:]

print("len(train_data): ", len(train_data))
print("len(test_data): ", len(test_data))
print("len(val_data): ", len(val_data))


len(train_data):  935
len(test_data):  110
len(val_data):  55


In [4]:
import torch
from torch.utils.data import Dataset, DataLoader

class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data
        self.encoded_texts = []
        for entry in data:
            request = format_input(entry)
            response = f"\n\\n### Response: \\n{entry['output']}"
            full_text = request + response
            self.encoded_texts.append(tokenizer.encode(full_text))
        
    def __getitem__(self, index):
        return self.encoded_texts[index]
    
    def __len__(self):
        return len(self.data)

In [5]:
def custom_collate(batch, ignore_index = -100,pad_token = 50256, allowed_max_length=None, device='cpu'):
    batch_max_length = max(len(item)+1  for item in batch)

    inputs, targets = [], []

    for item in batch:
        new_item = item.copy()
        new_item += [pad_token]

        padded = new_item + [pad_token] * (batch_max_length - len(new_item))
        input = torch.tensor(padded[:-1])
        target = torch.tensor(padded[1:])

        mask = target == pad_token
        indices = torch.nonzero(mask).squeeze()
        if indices.numel() > 1:
            target[indices[1:]] = ignore_index
        
        if allowed_max_length is not None:
            input = input[:allowed_max_length]
            target = target[:allowed_max_length]
        inputs.append(input)
        targets.append(target)

    input_tensor = torch.stack(inputs).to(device)
    target_tensor = torch.stack(targets).to(device)
    return input_tensor, target_tensor

input1 = [0, 1, 2, 3, 4]
input2 = [5, 6]
input3 = [7, 8, 9]

batch = (input1, input2, input3)
print(custom_collate(batch))

(tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]]), tensor([[    1,     2,     3,     4, 50256],
        [    6, 50256,  -100,  -100,  -100],
        [    8,     9, 50256,  -100,  -100]]))


In [14]:
from functools import partial
import tiktoken
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
custom_collate_fn = partial(custom_collate, device=device, allowed_max_length=1024)

num_workers = 0
batch_size = 8

torch.manual_seed(123)
tokenizer = tiktoken.get_encoding('gpt2')
train_dataset = InstructionDataset(train_data, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=batch_size, collate_fn=custom_collate_fn, shuffle=True, drop_last=True, num_workers=num_workers)

test_dataset = InstructionDataset(test_data, tokenizer)
test_loader = DataLoader(test_dataset, batch_size=batch_size, collate_fn=custom_collate_fn, shuffle=False, drop_last=False, num_workers=num_workers)

val_dataset = InstructionDataset(val_data, tokenizer)
val_loader = DataLoader(val_dataset, batch_size=batch_size, collate_fn=custom_collate_fn, shuffle=False, drop_last=False, num_workers=num_workers)


In [7]:
for input,output in train_loader:
    print("input: ", input.shape)
    print("output: ", output.shape)

input:  torch.Size([8, 67])
output:  torch.Size([8, 67])
input:  torch.Size([8, 80])
output:  torch.Size([8, 80])
input:  torch.Size([8, 79])
output:  torch.Size([8, 79])
input:  torch.Size([8, 74])
output:  torch.Size([8, 74])
input:  torch.Size([8, 71])
output:  torch.Size([8, 71])
input:  torch.Size([8, 76])
output:  torch.Size([8, 76])
input:  torch.Size([8, 86])
output:  torch.Size([8, 86])
input:  torch.Size([8, 73])
output:  torch.Size([8, 73])
input:  torch.Size([8, 68])
output:  torch.Size([8, 68])
input:  torch.Size([8, 81])
output:  torch.Size([8, 81])
input:  torch.Size([8, 66])
output:  torch.Size([8, 66])
input:  torch.Size([8, 74])
output:  torch.Size([8, 74])
input:  torch.Size([8, 73])
output:  torch.Size([8, 73])
input:  torch.Size([8, 81])
output:  torch.Size([8, 81])
input:  torch.Size([8, 73])
output:  torch.Size([8, 73])
input:  torch.Size([8, 83])
output:  torch.Size([8, 83])
input:  torch.Size([8, 75])
output:  torch.Size([8, 75])
input:  torch.Size([8, 72])
out

In [8]:
import sys
from pathlib import Path
sys.path.append(str(Path("..").resolve())) # result parent folder
from miniGPT.model import GPT2
from tools.gpt_download import download_and_load_gpt2

setting, params = download_and_load_gpt2(model_size="124M", models_dir='../gpt2')

# config from GPT2 model
model_config = {
    "vocab_size": 50257,
    "context_length": 1024,
    "embedding_dim": 768,
    "num_heads": 12,
    "num_layers": 12,
    "dropout": 0.1,
    "qkv_bias": True, # OPENAI CONFIG
}
model = GPT2(model_config)
model.load_pretrain_weights(params)

File already exists and is up-to-date: ../gpt2/124M/checkpoint
File already exists and is up-to-date: ../gpt2/124M/encoder.json
File already exists and is up-to-date: ../gpt2/124M/hparams.json
File already exists and is up-to-date: ../gpt2/124M/model.ckpt.data-00000-of-00001
File already exists and is up-to-date: ../gpt2/124M/model.ckpt.index
File already exists and is up-to-date: ../gpt2/124M/model.ckpt.meta
File already exists and is up-to-date: ../gpt2/124M/vocab.bpe
initializing multi head attention with d_in: 768 d_out: 768 num_heads: 12 context_length: 1024 dropout: 0.1
initializing multi head attention with d_in: 768 d_out: 768 num_heads: 12 context_length: 1024 dropout: 0.1
initializing multi head attention with d_in: 768 d_out: 768 num_heads: 12 context_length: 1024 dropout: 0.1
initializing multi head attention with d_in: 768 d_out: 768 num_heads: 12 context_length: 1024 dropout: 0.1
initializing multi head attention with d_in: 768 d_out: 768 num_heads: 12 context_length: 102

In [11]:
# reuse calc_loss_loader function from previous sections, 
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch = input_batch.to(device)
    target_batch = target_batch.to(device)
    logits = model(input_batch)
    loss = torch.nn.functional.cross_entropy(logits.flatten(0,1), target_batch.flatten())
    return loss

def calc_loss_loader(data_loader, model, device, num_batches = None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    
    if num_batches == None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))

    for i, (input,target) in enumerate(data_loader):
        if (i < num_batches):
            loss = calc_loss_batch(input, target, model, device)
            total_loss += loss
        else:
            break

    return total_loss/num_batches
    

In [12]:
# resuse train_model function from previous sections
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)

    model.train()
    return train_loss, val_loss



def train_model(model, train_loader, val_loader, optimizer, device, num_epochs,
                eval_freq, eval_iter, start_context, tokenizer):
    train_losses, val_losses, track_token_seen = [],[],[]
    token_seen, global_step = 0, -1
    for epoch in range(num_epochs):
        model.train()
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward()
            optimizer.step()
            
            token_seen += input_batch.numel()
            global_step += 1

            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(model, train_loader, val_loader, device, eval_iter)

                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_token_seen.append(token_seen)
                print(f"Ep {epoch + 1} (Step: {global_step}): ",
                      f"Train loss {train_loss} ",
                      f"val_loss: {val_loss}")
                
            # generate_and_print_sample(model, tokenizer, device, start_context)

    return train_losses, val_losses, track_token_seen

In [15]:
# we calculate loss before finetuning
model.to(device)
torch.manual_seed(123)

with torch.no_grad():
    train_loss = calc_loss_loader(train_loader, model, device, num_batches=5)
    val_loss = calc_loss_loader(val_loader, model, device, num_batches=5)

print("train_loss: ", train_loss)
print("val_loss: ", val_loss)

train_loss:  tensor(4.3951)
val_loss:  tensor(4.3315)


In [16]:
import time

start_time = time.time()
torch.manual_seed(123)
optimizer = torch.optim.AdamW(model.parameters(), lr = 0.0005, weight_decay=0.1)
num_epochs = 2

train_losses, val_losses, track_token_seen = train_model(model, train_loader, val_loader, optimizer, device, num_epochs, eval_freq = 5, eval_iter=5, start_context=format_input(val_data[0]), tokenizer=tokenizer)
end_time = time.time()
total_time = (end_time - start_time)/60
print(f"training time: {total_time:.2f} minutes")


Ep 1 (Step: 0):  Train loss 2.6684067249298096  val_loss: 2.6936447620391846
Ep 1 (Step: 5):  Train loss 1.1191492080688477  val_loss: 1.1466810703277588
Ep 1 (Step: 10):  Train loss 0.9562469720840454  val_loss: 1.1024153232574463
Ep 1 (Step: 15):  Train loss 0.8800069093704224  val_loss: 1.0689871311187744
Ep 1 (Step: 20):  Train loss 0.8964618444442749  val_loss: 1.0196776390075684
Ep 1 (Step: 25):  Train loss 0.755512535572052  val_loss: 0.9776951670646667
Ep 1 (Step: 30):  Train loss 0.7796028852462769  val_loss: 0.939623236656189
Ep 1 (Step: 35):  Train loss 0.7652277946472168  val_loss: 0.9051348567008972
Ep 1 (Step: 40):  Train loss 0.7471499443054199  val_loss: 0.9162314534187317
Ep 1 (Step: 45):  Train loss 0.8451302647590637  val_loss: 0.8999488949775696
Ep 1 (Step: 50):  Train loss 0.6948596239089966  val_loss: 0.8989917635917664
Ep 1 (Step: 55):  Train loss 0.7314487099647522  val_loss: 0.8837421536445618
Ep 1 (Step: 60):  Train loss 0.7393784523010254  val_loss: 0.8593884

In [17]:
torch.manual_seed(123)

for entry in test_data[:3]:        #1
    input_text = format_input(entry)

    generated_text = model.generate(          #2
        context=input_text,
        max_new_tokens=256,
        context_size=model_config["context_length"],
        eos_id=50256
    )

    response_text = (
        generated_text[len(input_text):]
        .replace("### Response:", "")
        .strip()
    )

    print(input_text)
    print(f"\nCorrect response:\n>> {entry['output']}")
    print(f"\nModel response:\n>> {response_text.strip()}")
    print("-------------------------------------")

Below is an instruction that describe a task.Write a response that appropriately completes the request.
\n ### Instruction: \nRewrite the sentence using a simile.
\n### Input: \nThe car is very fast.

Correct response:
>> The car is as fast as lightning.

Model response:
>> \n \nThe car is a snail.
-------------------------------------
Below is an instruction that describe a task.Write a response that appropriately completes the request.
\n ### Instruction: \nWhat type of cloud is typically associated with thunderstorms?

Correct response:
>> The type of cloud typically associated with thunderstorms is cumulonimbus.

Model response:
>> \n \nThe type of cloud is typically associated with thunderstorms.
-------------------------------------
Below is an instruction that describe a task.Write a response that appropriately completes the request.
\n ### Instruction: \nName the author of 'Pride and Prejudice'.

Correct response:
>> Jane Austen.

Model response:
>> \n \nThe author of 'Pride an

In [18]:
from tqdm import tqdm

for i, entry in tqdm(enumerate(test_data), total=len(test_data)):
    input_text = format_input(entry)

    generated_text = model.generate(          #2
        context=input_text,
        max_new_tokens=256,
        context_size=model_config["context_length"],
        eos_id=50256
    )

    response_text = (
        generated_text[len(input_text):]
        .replace("### Response:", "")
        .strip()
    )

    test_data[i]["model_response"] = response_text

with open("instruction-data-with-response.json", "w") as file:
    json.dump(test_data, file, indent=4)  #1

100%|██████████| 110/110 [02:23<00:00,  1.31s/it]


## Evaluate fine-tuned model using Llama model
Before run any code bellow first we need start Llama locally
```
$ ollama run llama3
```

In [21]:
import json
from tqdm import tqdm

file_path = "instruction-data-with-response.json"
with open(file_path, "r") as file:
    test_data = json.load(file)

def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = (
        f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""
    )

    return instruction_text + input_text

In [22]:
import json
import urllib.request

def query_model(
    prompt,
    model="llama3",
    url="http://localhost:11434/api/chat"
):
    data = {                     #1
        "model": model,
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "options": {             #2
            "seed": 123,
            "temperature": 0,
            "num_ctx": 2048
        }
    }

    payload = json.dumps(data).encode("utf-8")   #3

    request = urllib.request.Request(            #4
        url,
        data=payload,
        method="POST"
    )

    request.add_header("Content-Type", "application/json")  #4

    response_data = ""

    with urllib.request.urlopen(request) as response:       #5
        while True:
            line = response.readline().decode("utf-8")

            if not line:
                break

            response_json = json.loads(line)
            response_data += response_json["message"]["content"]

    return response_data

In [24]:
def generate_model_scores(json_data, json_key, model="llama3"):
    scores = []

    for entry in tqdm(json_data, desc="Scoring entries"):
        prompt = (
            f"Given the input `{format_input(entry)}` "
            f"and correct output `{entry['output']}`, "
            f"score the model response `{entry[json_key]}` "
            f"on a scale from 0 to 100, where 100 is the best score. "
            f"Respond with the integer number only."    #1
        )

        score = query_model(prompt, model)

        try:
            scores.append(int(score))
        except ValueError:
            print(f"Could not convert score: {score}")
            continue

    return scores

In [25]:
scores = generate_model_scores(test_data, 'model_response')
print(f"Number of scores: {len(scores)} of {len(test_data)}")
print(f"Average score: {sum(scores)/len(scores):.2f}\n")

Scoring entries:  45%|████▌     | 50/110 [00:38<00:42,  1.41it/s]

Could not convert score: Commence
85


Scoring entries: 100%|██████████| 110/110 [01:23<00:00,  1.31it/s]

Number of scores: 109 of 110
Average score: 31.46

